# Road Surface Classification — Vibration Embedding & Clustering

## 1. Install & Imports <a id='1'></a>

In [1]:
import glob
import pandas as pd
import numpy as np
import os
import glob
import re
import matplotlib.pyplot as plt
import random
from pathlib import Path
from tqdm import tqdm
from scipy.fft import fft
import scipy.stats as stats

# Config

In [6]:
CONFIG = {
    'data_dir': Path('../Datasets/Processed_Data/Labeled_Data_Without_GPS'),
    'window_size': 1024,
    'overlap': 0.5,
    'pca_variance_threshold': 0.95
}

# Loading Data

In [7]:
def extract_surface_type_id(path):
    match = re.search(r'SurfaceTypeID_(\d+)', path)
    return int(match.group(1)) if match else None

data_dir = CONFIG['data_dir']
file_paths = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

files_df = pd.DataFrame({
    "full_path": file_paths,
    "filename": [os.path.basename(p) for p in file_paths],
})

files_df["surface_id"] = files_df["full_path"].apply(extract_surface_type_id)

print(files_df["surface_id"].value_counts())

files_df.head(2)

surface_id
10    406
9     403
8      98
7      96
6      90
5      66
2      57
3      40
4      30
11     28
12     20
1      10
Name: count, dtype: int64


,full_path,filename,surface_id
0,../Datasets/Processed_Data/Labeled_Data_Withou...,2019-02-15_SurfaceTypeID_9_SamsungGalaxyJ7_exp...,9
1,../Datasets/Processed_Data/Labeled_Data_Withou...,2019-09-02_SurfaceTypeID_9_SamsungGalaxyS7_exp...,9


In [8]:
# Load and print head of first data file
first_file_path = files_df.iloc[0]['full_path']
first_data = pd.read_csv(first_file_path)
print(first_data.head())

  sensorName    valueX    valueY    valueZ     timestamp
0  Gyroscope -0.001634 -0.007524 -0.000831  1.550261e+12
1  Gyroscope  0.002392 -0.005621 -0.003298  1.550261e+12
2  Gyroscope  0.005251 -0.006566 -0.004236  1.550261e+12
3  Gyroscope  0.005728 -0.010090 -0.004081  1.550261e+12
4  Gyroscope  0.003947 -0.011895 -0.003882  1.550261e+12


# Embedding Feature extraction using TS2Vec

In [9]:

# Feature extraction code goes here (not shown in the provided snippets)

def zero_crossing_rate(signal):
    return ((signal[:-1] * signal[1:]) < 0).sum()

acc_windows = []
acc_labels = []

gyro_windows = []
gyro_labels = []

WINDOW_SIZE = CONFIG['window_size']
STEP_SIZE = int(WINDOW_SIZE * (1 - CONFIG['overlap']))

for idx, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Processing files"):
    file_path = row['full_path']
    surface_id = row['surface_id']
    
    data_df = pd.read_csv(file_path)

    # Pad if data is shorter than one window
    if len(data_df) < WINDOW_SIZE:
        pad_size = WINDOW_SIZE - len(data_df)
        pad_df   = pd.concat([data_df] * (pad_size // len(data_df) + 1)).iloc[:pad_size]
        data_df  = pd.concat([data_df, pad_df], ignore_index=True)

    # Pad last incomplete window with edge values
    remainder = len(data_df) % STEP_SIZE
    if remainder != 0:
        pad_size = WINDOW_SIZE - remainder
        pad_df   = data_df.iloc[-pad_size:].copy()
        data_df  = pd.concat([data_df, pad_df], ignore_index=True)

    all_windows = []
    all_labels = []
    for start in range(0, len(data_df) - WINDOW_SIZE + 1, STEP_SIZE):
        window = data_df[start:start + WINDOW_SIZE]

        xyz = window[['valueX', 'valueY', 'valueZ']].values  # (WINDOW_SIZE, 3)
        all_windows.append(xyz)
        all_labels.append(row['surface_id'])


    if 'accelerometer' in file_path.lower():
        acc_windows.extend(all_windows)
        acc_labels.extend(all_labels)
    elif 'gyroscope' in file_path.lower():
        gyro_windows.extend(all_windows)
        gyro_labels.extend(all_labels)

print(f"Extracted windows from {len(acc_windows)} accelerometer windows and {len(gyro_windows)} gyroscope windows.")

Processing files: 100%|██████████| 1344/1344 [00:05<00:00, 246.85it/s]

Extracted windows from 7962 accelerometer windows and 7890 gyroscope windows.


In [10]:
import torch
from ts2vec import TS2Vec

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

print(f"Using device: {device}")

acc_windows_np = np.array(acc_windows, dtype=np.float32)
gyro_windows_np = np.array(gyro_windows, dtype=np.float32)

acc_model = TS2Vec(input_dims=3, device=device, output_dims=128)
acc_loss_log = acc_model.fit(acc_windows_np, n_epochs=20, verbose=True)   # trains on ALL windows

gyro_model = TS2Vec(input_dims=3, device=device, output_dims=128)
gyro_loss_log = gyro_model.fit(gyro_windows_np, n_epochs=50, verbose=True)


torch.save(acc_model.net.state_dict(), 'ts2vec_accelerometer.pth')


Using device: mps


KeyboardInterrupt: 

In [ ]:
# Plot training loss logs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc_loss_log, linewidth=2)
axes[0].set_title('Accelerometer - TS2Vec Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(gyro_loss_log, linewidth=2)
axes[1].set_title('Gyroscope - TS2Vec Training Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()